In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.3 MB/s eta 0:00:00


# Importing the Libraries and Data loading

In [3]:
import numpy as np
import pandas as pd

import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

train_cc = train.copy()
test_cc = test.copy()
ss_cc = sample_submission.copy()

In [4]:
print("The shape of the train: ", train.shape)
print("The shape of the copied train: ", train_cc.shape)
print("The shape of the test: ", test.shape)
print("The shape of the copied test: ", train_cc.shape)

The shape of the train:  (2000, 8)
The shape of the copied train:  (2000, 8)
The shape of the test:  (500, 7)
The shape of the copied test:  (2000, 8)


# Creating the knowledge base

In [5]:
print("Creating Knowledge Base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]          # A, B, C, D or E
    correct_option = str(row[correct_letter])
    kb.append(correct_option)

print("Knowledge Base successfully created!")
print("Total documents:", len(kb))

Creating Knowledge Base...
Knowledge Base successfully created!
Total documents: 2000


In [6]:
print(kb[:5])
print(len(kb))

['Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Reddening', 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.', 'Simultaneity is relative, meaning that two events t

# Loading embedding model

In [7]:
print("Loading embedding model")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading embedding model


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


# Generating embeddings 

In [8]:
print("Generating embeddings for the knowledge base")

kb_embeddings = model.encode(kb,show_progress_bar=True)
print("Embeddings created.")
print("Embedding shape:", kb_embeddings.shape)

Generating embeddings for the knowledge base


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embeddings created.
Embedding shape: (2000, 384)


# Building the FAISS index

In [9]:
print("creating FAISS index.")
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("FAISS index created successfully")
print("Number of indexed documents: ", index.ntotal)

creating FAISS index.
FAISS index created successfully
Number of indexed documents:  2000


# Loading the zero-shot classifier For Q1, Q2, Q6

In [10]:
zs = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")
print("Zero-shot classifier loaded")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Zero-shot classifier loaded


In [11]:
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

print("Ground truth answer: ")
print(ans_150)

Ground truth answer: 
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


# Zero-shot classifier 

# Q1

Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)


In [12]:
result = zs(prompt_150, candidate_labels = labels_150)

result

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.',
 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
  'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
  'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."',
  'The butterfly effect is the phenomenon that a small change in the initial c

In [13]:
ground_truth_score = result["scores"][
    result["labels"].index(ans_150)
]

print("Ground-truth probablity: ", round(ground_truth_score, 3))

Ground-truth probablity:  0.384


# Q2

Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

## Embeding the prompt

In [14]:
query_embedding = model.encode([prompt_150])

In [15]:
query_embedding.shape

(1, 384)

## Searching the FAISS index

In [16]:
k = 10
distances, indices = index.search(query_embedding, k)
print(indices)

[[ 663 1701 1269 1532  576  847 1693 1906  168  150]]


## Rank of document 150

In [17]:
true_doc = 150
rank = list(indices[0]).index(true_doc) + 1
print("Rank of the document: ", rank)

Rank of the document:  10


# Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)

# Q3
Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

## Loading the Cross-Encoder

In [18]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-Encode loaded successfully.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encode loaded successfully.


In [19]:
retrieved_indices = indices[0]
print(retrieved_indices)

[ 663 1701 1269 1532  576  847 1693 1906  168  150]


## Retrieving the corresponding doc 

In [20]:
docs_10 = [kb[i] for i in retrieved_indices]
len(docs_10)

10

## creating Question-Document pairs

In [21]:
pairs = [[prompt_150, doc] for doc in docs_10]
len(pairs)

10

## Predicting Cross_encoder scores 

In [22]:
ce_scores = cross_encoder.predict(pairs)
print(ce_scores)

[4.6602316 4.6602316 4.7375226 4.7375226 4.6870446 4.7525764 4.7525764
 4.7525764 4.7072477 4.758512 ]


In [23]:
ranking = sorted(zip(retrieved_indices, ce_scores), key=lambda x: x[1], reverse = True)

## The rankings are --

In [24]:
for rank, (doc_id, score) in enumerate(ranking, start=1):
    print(
        f"Rank {rank}: "
        f"Document {doc_id}, "
        f"Score = {score:.4f}"
    )

Rank 1: Document 150, Score = 4.7585
Rank 2: Document 847, Score = 4.7526
Rank 3: Document 1693, Score = 4.7526
Rank 4: Document 1906, Score = 4.7526
Rank 5: Document 1269, Score = 4.7375
Rank 6: Document 1532, Score = 4.7375
Rank 7: Document 168, Score = 4.7072
Rank 8: Document 576, Score = 4.6870
Rank 9: Document 663, Score = 4.6602
Rank 10: Document 1701, Score = 4.6602


In [25]:
true_doc = 150

for rank, (doc_id, score) in enumerate(ranking, start=1):
    if doc_id == true_doc:
        print("True document rank: ", rank)
        break

True document rank:  1


# Q4
Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

## Getting  and embedding row 42

In [26]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

query_embedding_42 = model.encode([prompt_42])

In [27]:
# Retrieve the top 5 documents
k = 5
distances_42, indices_42 = index.search(query_embedding_42, k)

retrieved_indices_42 = indices_42[0]
print(retrieved_indices_42)

[241 439 456 506 605]


In [28]:
docs_5 = [kb[i] for i in retrieved_indices_42]
concatenated_docs = " ".join(docs_5)
final_text = (
    f"Context: {concatenated_docs}"
    f"Question: {prompt_42}"
)

In [29]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [30]:
tokens = tokenizer(final_text, truncation = False)

total_tokens = len(tokens["input_ids"])
print("Total Tokens: ", total_tokens)

Total Tokens:  216


# Q5
Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [31]:
# Retrieving the true document
true_document = kb[150]

## Create the RAG prompt 

In [32]:
rag_prompt = (
    f"Context: {true_document}"
    f"Question: {prompt_150}"
)

## Running the zero-shot classifier and extracting the Ground-Truth score

In [33]:
rag_result = zs(rag_prompt, candidate_labels = labels_150)
ground_truth_score = rag_result["scores"][
    rag_result["labels"].index(ans_150)
]

print("Ground-truth probablity:", round(ground_truth_score, 3))

Ground-truth probablity: 0.986


# Concept: The Danger of Bad Retrieval (Adversarial RAG) 

# Q6
What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [34]:
wrong_document = kb[999]
adversarial_prompt = (
    f"Context: {wrong_document}"
    f"Question: {prompt_150}"
)

## Running zero-shot classifier and getting ground_truth_score to find probablity of the correct option

In [35]:
adversarial_result = zs(adversarial_prompt, candidate_labels = labels_150)
ground_truth_score = adversarial_result["scores"][
    adversarial_result["labels"].index(ans_150)
]

print("Groung-truth probability: ", round(ground_truth_score, 3))

Groung-truth probability:  0.528


# Q7
For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

### What is Hit in this question??

1. First we need to get the first 100 rows of train.csv
2. then taking the prompts we need to embed using `all-MiniLm-L6-v2`
3. after retrieving the top 5 documents from `FAISS` we need to check whether the exact correct answer text for the row appears among those retrieved documents
4. if yes --> Hit
5. otherwise didn't count

In [36]:
hits = 0
k = 5

for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    correct_answer = str(row[row["answer"]])
    query_embedding = model.encode([prompt])

    distances, indices = index.search(query_embedding, k)
    retrieved_docs = [kb[idx] for idx in indices [0]]
    if any(correct_answer in doc for doc in retrieved_docs):
        hits +=1

hit_rate = (hits/100)*100

print("Hits: ", hits)
print("Hit Rate: ", round(hit_rate, 1), "%")

Hits:  73
Hit Rate:  73.0 %


# Q8
Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).



In [37]:
def map_at_3(actual, predicted):
    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)
    return 0.0

scores = []

k = 5

for i in range(20):
    row = train.iloc[i]

    prompt = str(row["prompt"])

    options = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    candidate_labels = list(options.values())
    correct_letter = row["answer"]

    query_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )

    distances, indices = index.search(query_embedding, k)

    retrieved_indices = indices[0]
    retrieved_docs = [kb[idx] for idx in retrieved_indices]

    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_document = retrieved_docs[np.argmax(ce_scores)]

    rag_prompt = (
        f"Context: {best_document} "
        f"Question: {prompt}"
    )


    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels
    )

    predicted_letters = []

    for label in result["labels"][:3]:
        for letter, option_text in options.items():
            if label == option_text:
                predicted_letters.append(letter)
                break

    score = map_at_3(correct_letter, predicted_letters)

    scores.append(score)


final_map3 = np.mean(scores)

print("Average MAP@3:", round(final_map3, 3))

Average MAP@3: 0.975
